In [1]:
import itertools
import logging
import os

import mlflow

from model.train_models import train_evaluate_model
from utils.data_prep import get_clean_combined_data

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

mlflow.sklearn.autolog(disable=True)

DOWNLOAD = False  # Set to use local data for testing

In [2]:
completed_runs_file = "completed_runs.txt"

if os.path.exists(completed_runs_file):
    with open(completed_runs_file, "r") as f:
        completed_runs = {line.strip() for line in f if line.strip()}
else:
    completed_runs = set()

print(f"Loaded {len(completed_runs)} completed runs from memory.")

Loaded 44 completed runs from memory.


In [3]:
ks = [0.25, 0.5, 0.75, 1]
ns = [4, 5]
event_cols = ["sub_event_type", "event_type"]
remove_abyei_options = [False]  # TODO Abyei currently not present for rainfall
include_food_options = [True, False]
include_rain_options = [True, False]
include_text_options = [True, False]

In [4]:
xgb_params = {
    "max_depth": [3, 5, 7],
    "min_child_weight": [1, 3, 5],
    "max_delta_step": [0, 1, 5],
    "gamma": [0, 1, 3, 5],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_alpha": [0, 0.1, 1, 2],
    "reg_lambda": [1, 5, 10],
    "colsample_bylevel": [0.6, 0.8, 1.0],
}

In [6]:
data_configs = itertools.product(
    remove_abyei_options,
    include_food_options,
    include_rain_options,
    include_text_options,
    ks,
    event_cols,
)

for (
    remove_abyei,
    include_food,
    include_rain,
    include_text,
    k,
    event_col,
) in data_configs:
    food_str = "_food" if include_food else ""
    rain_str = "_rain" if include_rain else ""
    text_str = "_text" if include_text else ""
    abyei_str = "_remove_abyei" if remove_abyei else ""
    event_str = "event" if event_col == "event_type" else "sub"

    which_data = f"acled_{event_str}{food_str}{rain_str}{text_str}{abyei_str}"

    if include_text:
        pca_options = [
            True
        ]  # Run both when text is included #TODO this is temp changed !!!!!!!!!!!!!
    else:
        pca_options = [False]  # Only run without PCA when text isn't included

    all_runs_completed = True
    for n in ns:
        for use_pca in pca_options:
            pca_str = "_pca" if use_pca else ""
            expected_run = f"{which_data}{pca_str}_{k}_{n}"
            if expected_run not in completed_runs:
                all_runs_completed = False
                break  # Stop checking this inner loop if we find a missing run
        if not all_runs_completed:
            break  # Stop checking the outer loop too

    if all_runs_completed:
        print(
            f"Skipping data load for {which_data} - all associated runs are complete."
        )
        continue

    # Only load data if I have at least one missing run
    data_sources = [
        src
        for src, include in zip(
            ["food", "rain", "text"], [include_food, include_rain, include_text]
        )
        if include
    ]

    model_data, predictor_cols = get_clean_combined_data(
        data_sources=data_sources,
        download=DOWNLOAD,
        remove_abyei=remove_abyei,
        k=k,
        event_col=event_col,
    )

    for n in ns:
        for use_pca in pca_options:
            pca_str = "_pca" if use_pca else ""
            run_name = f"{which_data}{pca_str}_{k}_{n}"

            if run_name in completed_runs:
                print(f"Skipping already completed run: {run_name}")
                continue

            all_params = {
                **xgb_params,
                "k": k,
                "event_col": event_col,
                "remove_abyei": remove_abyei,
                "n_splits": n,
                "use_pca": use_pca,
            }

            with mlflow.start_run(run_name=run_name):
                mlflow.set_tags(
                    {
                        "data_version": which_data,
                        "remove_abyei": remove_abyei,
                        "include_food": include_food,
                        "include_rain": include_rain,
                        "include_text": include_text,
                        "use_pca": use_pca,
                        "k": k,
                        "n_splits": n,
                        "event_col": event_col,
                    }
                )
                logger.info(f"Running mode: {run_name}")

                results, best_params = train_evaluate_model(
                    model_data,
                    predictor_cols,
                    all_params,
                    best_params=False,
                    use_pca=use_pca,
                )

                mlflow.log_params(best_params)
                mlflow.log_metrics({key: float(val) for key, val in results.items()})
                mlflow.log_dict(results, "model_report.json")

                completed_runs.add(run_name)  # Add to log file
                with open(completed_runs_file, "a") as f:
                    f.write(run_name + "\n")

Skipping data load for acled_sub_food_rain_text - all associated runs are complete.
Skipping data load for acled_event_food_rain_text - all associated runs are complete.
Skipping data load for acled_sub_food_rain_text - all associated runs are complete.
Skipping data load for acled_event_food_rain_text - all associated runs are complete.
Skipping data load for acled_sub_food_rain_text - all associated runs are complete.
Skipping data load for acled_event_food_rain_text - all associated runs are complete.
Skipping data load for acled_sub_food_rain_text - all associated runs are complete.
Skipping data load for acled_event_food_rain_text - all associated runs are complete.
Skipping data load for acled_sub_food_rain - all associated runs are complete.
Skipping data load for acled_event_food_rain - all associated runs are complete.
Skipping data load for acled_sub_food_rain - all associated runs are complete.
Skipping data load for acled_event_food_rain - all associated runs are complete.


INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 0.5 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:Train model:download=False: Reading local file ../data/hdx\Sudan - Food Prices.csv
INFO:Train model:download=False: Reading local file ../data/hdx\South Sudan - Food Prices.csv
INFO:Data preparation:Food prices data processed.
INFO:Data preparation:Notes data processed.
INFO:__main__:Running mode: acled_sub_food_text_pca_0.5_4
INFO:Text processing:PCA fit on train embeddings only: 768 raw dims -> 42 components for 90% variance.
INFO:Cross validation:Cross-validation testing splits:


--- Fold 1 ---
Train window: 2018-01 to 2018-12 (228 rows)
Test window:  2019-01 to 2019-12 (228 rows)
------------------------------
--- Fold 2 ---
Train window: 2018-01 to 2019-12 (456 rows)
Test window:  2020-01 to 2020-12 (228 rows)
------------------------------
--- Fold 3 ---
Train window: 2018-01 to 2020-12 (684 rows)
Test window:  2021-01 to 2021-12 (228 rows)
------------------------------
--- Fold 4 ---
Train window: 2018-01 to 2021-12 (912 rows)
Test window:  2022-01 to 2022-12 (228 rows)
------------------------------


INFO:__main__:Running mode: acled_sub_food_text_pca_0.5_5
INFO:Text processing:PCA fit on train embeddings only: 768 raw dims -> 42 components for 90% variance.
INFO:Cross validation:Cross-validation testing splits:


--- Fold 1 ---
Train window: 2018-01 to 2018-10 (190 rows)
Test window:  2018-11 to 2019-08 (190 rows)
------------------------------
--- Fold 2 ---
Train window: 2018-01 to 2019-08 (380 rows)
Test window:  2019-09 to 2020-06 (190 rows)
------------------------------
--- Fold 3 ---
Train window: 2018-01 to 2020-06 (570 rows)
Test window:  2020-07 to 2021-04 (190 rows)
------------------------------
--- Fold 4 ---
Train window: 2018-01 to 2021-04 (760 rows)
Test window:  2021-05 to 2022-02 (190 rows)
------------------------------
--- Fold 5 ---
Train window: 2018-01 to 2022-02 (950 rows)
Test window:  2022-03 to 2022-12 (190 rows)
------------------------------


KeyboardInterrupt: 